# Model Development (Phase 5)

> Goal: compare baseline models, document selection logic, and connect offline metrics to production policy decisions.

This notebook is intentionally artifact-aware: it reads outputs from the training pipeline (`scripts/train_model.py`) so analysis is reproducible and fast.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Resolve project root from notebook location.
PROJECT_ROOT = Path.cwd().resolve().parent

metrics_dir = PROJECT_ROOT / "reports" / "metrics"
figures_dir = PROJECT_ROOT / "reports" / "figures" / "evaluation"

sns.set_theme(style="whitegrid")
metrics_dir, figures_dir

## Load Model Comparison Outputs

These files are generated by `python scripts/train_model.py` and are the source of truth for Phase 5 model ranking.

In [ ]:
comparison_path = metrics_dir / "model_comparison.csv"
strategy_path = metrics_dir / "model_decision_strategy.json"
evaluation_path = metrics_dir / "evaluation_summary.csv"

comparison_df = pd.read_csv(comparison_path)
evaluation_df = pd.read_csv(evaluation_path)
strategy = json.loads(strategy_path.read_text(encoding="utf-8"))

comparison_df

In [ ]:
display_cols = ["model_name", "cv_roc_auc", "test_roc_auc", "test_recall", "test_precision", "test_f1_score"]
comparison_view = comparison_df[display_cols].copy()
comparison_view = comparison_view.sort_values("test_roc_auc", ascending=False).reset_index(drop=True)
comparison_view

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=comparison_df.sort_values("test_roc_auc", ascending=False),
    x="model_name",
    y="test_roc_auc",
    hue="model_name",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Test ROC-AUC by Model")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("ROC-AUC")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(
    data=comparison_df.sort_values("test_recall", ascending=False),
    x="model_name",
    y="test_recall",
    hue="model_name",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Test Recall by Model")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Recall")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## Policy Decision Layer

The training stage exports an explicit strategy file that supports business-specific operating modes:
- primary baseline model,
- recommended threshold for recall-focused campaigns,
- high-recall challenger model for aggressive retention.

In [ ]:
policy_df = pd.json_normalize(strategy, sep=".")
policy_df.T.rename(columns={0: "value"})

In [ ]:
eval_view = evaluation_df[["model_label", "strategy_label", "threshold", "precision", "recall", "f1_score", "roc_auc"]].copy()
eval_view.sort_values(["model_label", "strategy_label"]).reset_index(drop=True)

## Summary

- Gradient Boosting leads on ranking quality (ROC-AUC) and remains the primary baseline.
- Policy thresholding materially improves recall for retention-oriented campaigns.
- SVM remains a credible challenger when business strategy tolerates lower precision for broader outreach.

For productionized runs, use the CLI scripts rather than re-implementing logic in notebooks.